# Сравнение результативности разных моделей
Так же используем 41 бинарную классификацию для решения задачи. Категории все разные и могут быть обусловлены разным набором признаков. Поэтому для каждой категории будем рассчитывать результативность предсказания на разных моделях. Так мы поймем, для каких категорий какая модель лучше подходит.

## Загрузка данных

In [ ]:
conda create -n pycaret_env python=3.10 -y

Channels:
 - defaults
Platform: osx-64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 25.5.1
    latest version: 25.7.0

Please update conda by running

    $ conda update -n base -c defaults conda



## Package Plan ##

  environment location: /opt/anaconda3/envs/pycaret_env

  added / updated specs:
    - python=3.10


The following NEW packages will be INSTALLED:

  bzip2              pkgs/main/osx-64::bzip2-1.0.8-h6c40b1e_6 
  ca-certificates    pkgs/main/osx-64::ca-certificates-2025.7.15-hecd8cb5_0 
  expat              pkgs/main/osx-64::expat-2.7.1-h6d0c2b6_0 
  libcxx             pkgs/main/osx-64::libcxx-19.1.7-haebbb44_3 
  libffi             pkgs/main/osx-64::libffi-3.4.4-hecd8cb5_1 
  ncurses            pkgs/main/osx-64::ncurses-6.5-h923df54_0 
  openssl            pkgs/main/osx-64::openssl-3.0.17-hee2dfae_0 
  pip                pkgs/main/noarch::pip-26.0.1-pyhc872135_0 
  python             pkgs/main/osx-64::python-3.10.18

In [1]:
import polars as pl
import polars.selectors as cs

import numpy as np
import matplotlib.pyplot as plt

from catboost import Pool, CatBoostClassifier
from sklearn.metrics import roc_auc_score

from pycaret.classification import ClassificationExperiment

In [2]:
import sklearn
import pycaret
print(sklearn.__version__)
print(pycaret.__version__)

1.4.2
3.3.2


In [5]:
data_path = 'data'

In [6]:
%%time

train = (
    pl
        .scan_parquet(data_path+'/train'+'/train_main_features.parquet')
        # .join(pl.scan_parquet(data_path+'train_extra_features.parquet'), on="customer_id")
        .with_columns(
            cs
                .by_dtype(pl.Float64)
                .cast(pl.Float32)
                # .fill_null(-99999)
        )
        .with_columns(
            cs
                .starts_with("cat_feature_")
                .cast(pl.Int32)
                # .fill_null(-1)
                # .cast(pl.Categorical)
        )
        .collect()
)

train

CPU times: user 968 ms, sys: 3.05 s, total: 4.01 s
Wall time: 1.36 s


customer_id,cat_feature_1,cat_feature_2,cat_feature_3,cat_feature_4,cat_feature_5,cat_feature_6,cat_feature_7,cat_feature_8,cat_feature_9,cat_feature_10,cat_feature_11,cat_feature_12,cat_feature_13,cat_feature_14,cat_feature_15,cat_feature_16,cat_feature_17,cat_feature_18,cat_feature_19,cat_feature_20,cat_feature_21,cat_feature_22,cat_feature_23,cat_feature_24,cat_feature_25,cat_feature_26,cat_feature_27,cat_feature_28,cat_feature_29,cat_feature_30,cat_feature_31,cat_feature_32,cat_feature_33,cat_feature_34,cat_feature_35,cat_feature_36,…,num_feature_96,num_feature_97,num_feature_98,num_feature_99,num_feature_100,num_feature_101,num_feature_102,num_feature_103,num_feature_104,num_feature_105,num_feature_106,num_feature_107,num_feature_108,num_feature_109,num_feature_110,num_feature_111,num_feature_112,num_feature_113,num_feature_114,num_feature_115,num_feature_116,num_feature_117,num_feature_118,num_feature_119,num_feature_120,num_feature_121,num_feature_122,num_feature_123,num_feature_124,num_feature_125,num_feature_126,num_feature_127,num_feature_128,num_feature_129,num_feature_130,num_feature_131,num_feature_132
i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
1000001,1,0,2,1,2,3,2,2,4,2,0,0,0,1,2,0,2,2,2,0,0,0,2,0,0,0,1,2,1,2,2,0,0,212,0,0,…,-0.284519,null,-0.004499,null,-0.050159,-0.002297,-0.043592,-0.061488,0.450342,null,0.0,null,-0.060492,-0.279105,null,-0.429813,null,-0.009654,-0.293036,null,-0.493959,-0.019079,null,null,null,null,-0.001357,-0.031281,-0.046146,null,-0.445279,null,null,-0.107666,-0.418616,null,null
1000002,1,0,0,1,0,3,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,212,1,0,…,-0.284519,0.460275,-0.004278,null,-0.556244,-0.002297,-0.043592,-0.061488,1.998252,-0.009552,0.0,-0.005762,-0.060492,-0.279105,null,-0.429813,null,-0.009654,-0.293036,-0.004421,-0.256445,-0.014154,null,-0.24167,null,null,-0.001357,-0.031281,-0.046146,-0.10217,1.550722,null,null,-0.170724,-0.805771,-0.397803,-0.373734
1000003,1,0,0,1,0,3,0,0,4,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,1,0,0,0,0,0,0,212,0,0,…,-0.284519,-0.264397,-0.004278,null,null,-0.002297,-0.043592,-0.061488,-0.264078,null,0.0,-0.26649,-0.060492,-0.279105,null,-0.429813,null,-0.009654,-0.293036,-0.004421,-0.57313,-0.019124,null,-0.24167,null,null,-0.001357,-0.031281,-0.046146,null,-0.475778,null,null,-0.170724,-0.602005,-0.397803,-0.373734
1000004,1,0,2,1,2,3,2,2,3,2,0,0,0,1,2,2,2,2,2,0,0,2,2,0,0,0,0,2,1,2,2,0,0,212,0,0,…,-0.284519,null,-0.004499,null,null,-0.002297,-0.043592,-0.061488,0.688482,-0.009552,0.0,null,-0.060492,null,null,-0.429813,null,-0.009654,-0.293036,null,-0.57313,null,null,-0.505441,null,0.714631,-0.001357,-0.031281,-0.046146,null,-0.475778,0.111196,0.116695,null,-0.724265,null,null
1000005,1,2,0,1,0,3,0,0,2,1,2,2,2,0,1,0,0,0,0,2,2,2,0,2,2,2,2,0,0,0,0,2,2,212,2,2,…,-0.284519,-0.264397,-0.004499,null,null,-0.002297,-0.043592,null,-0.264078,null,0.0,-0.26649,null,-0.279105,null,-0.429813,null,-0.009654,-0.293036,-0.004421,-0.57313,-0.018674,null,null,null,null,null,null,-0.046146,null,null,null,null,-0.107666,null,-0.397803,-0.373734
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1749996,1,0,2,0,2,3,2,2,4,2,0,0,0,1,2,0,2,2,2,0,0,2,2,0,0,0,0,2,1,2,2,0,0,212,0,0,…,null,null,-0.004499,-0.150243,null,null,null,2.831928,null,null,null,null,-0.060492,null,null,null,null,null,null,null,null,-0.019236,null,-0.505441,null,null,-0.001357,-0.031281,null,null,-0.475778,null,null,null,0.335315,null,null
1749997,1,0,2,1,2,3,2,2,2,2,0,0,0,1,2,0,2,2,2,0,0,0,2,0,0,0,0,2,1,2,2,0,0,212,1,0,…,-0.284519,null,-0.004499,null,-0.510019,-0.002297,-0.043592,-0.061488,-0.264078,-0.009552,0.0,null,-0.060492,-0.279105,null,2.155184,n

In [9]:
target = (
    pl
        .read_parquet(data_path+'/baseline'+'/train_target.parquet')
        .with_columns(
            cs
                .starts_with("target_")
                .cast(pl.Boolean)
        )
)

target

customer_id,target_1_1,target_1_2,target_1_3,target_1_4,target_1_5,target_2_1,target_2_2,target_2_3,target_2_4,target_2_5,target_2_6,target_2_7,target_2_8,target_3_1,target_3_2,target_3_3,target_3_4,target_3_5,target_4_1,target_5_1,target_5_2,target_6_1,target_6_2,target_6_3,target_6_4,target_6_5,target_7_1,target_7_2,target_7_3,target_8_1,target_8_2,target_8_3,target_9_1,target_9_2,target_9_3,target_9_4,target_9_5,target_9_6,target_9_7,target_9_8,target_10_1
i32,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool
1000001,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,true,false,false,false
1000002,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,true,false,false,false,false,false,false,false,false,false,false,false
1000003,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,true
1000004,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,true,false,false,false,false,false,false,true,false,false,false
1000005,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,true
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1749996,false,false,false,false,false,false,false,false,false,false,false,false,false,true,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,true,false,false,false,false,false,false
1749997,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,true,false,false
1749998,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,true


In [10]:
cat_feature_names = [col_name for col_name in train.columns if col_name.startswith("cat_feature")]
#cat_feature_names

CPU times: user 14min, sys: 47 s, total: 14min 47s

Wall time: 2h 17min 15s

## Обучение моделей

In [11]:
%%time

limit = 100_000 #если компьютер шустрый ставь 750_000 
minfrac = 0.1 #Желаемая доля в выборке 
s = ClassificationExperiment()
stats=[]

for target_name in target.columns[1:]:
    print("="*60,target_name)
    
    data = train.sample(limit, shuffle = True, seed = 7
                    ).join(target.select(["customer_id",target_name]),
                              on="customer_id"
                             )#.to_pandas()
    if data[target_name].sum()/limit<minfrac:
        
        data1 = target.select(["customer_id",target_name]
                             ).filter(pl.col(target_name)>0
                                     ).sample( min(round(limit*minfrac), target[target_name].sum()), 
                                              shuffle = True, seed = 7 
                                             )
        data0 = target.select(["customer_id",target_name]
                             ).filter(pl.col(target_name)==0
                                     ).sample( limit-data1.shape[0], 
                                              shuffle = True, seed = 7 
                                             )
        print(data[target_name].sum(), round(limit*minfrac), data1.shape[0], data0.shape[0])
        data = train.sample(limit, 
                             shuffle = True, 
                             seed = 7
                            ).join(pl.concat([data0, data1]),
                                      on="customer_id"
                                     ).drop("customer_id")
    data = data.to_pandas()
    
    #заплатка на глюк polars при переносе категориальных колонок в pandas 
    data[cat_feature_names] = data[cat_feature_names].astype("str")
    
    try:
        s.setup(data, 
            target = target_name, 
            log_experiment=False, 
            experiment_name=target_name
            #use_gpu=True,
           )
        best = s.compare_models(sort = 'AUC')

        leaderboard = s.pull()
        leaderboard.to_csv(f'leaderboard_{target_name}_{limit=}_{minfrac=}')
        stats.append([target_name, 
                      leaderboard.iloc[0].Model,leaderboard.iloc[0].AUC, 
                      leaderboard.iloc[1].Model,leaderboard.iloc[1].AUC, 
                      leaderboard.iloc[2].Model,leaderboard.iloc[2].AUC
                     ]
                    )
        #print(stats)

        s.save_model(best, f'caret_{target_name}_{limit=}_{minfrac=}')
        #break
    except:
        print("Ашипка.")
    
stats = pl.DataFrame(stats, schema=["tcol","model","AUC","model2","AUC2","model3","AUC3"])
stats

============================================================ target_1_1
1045 10000 7797 92203


,Description,Value
0,Session id,5677
1,Target,target_1_1
2,Target type,Binary
3,Original data shape,"(13505, 200)"
4,Transformed data shape,"(13505, 314)"
5,Transformed train set shape,"(9453, 314)"
6,Transformed test set shape,"(4052, 314)"
7,Numeric features,132
8,Categorical features,67
9,Rows with missing values,100.0%


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
catboost,CatBoost Classifier,0.9287,0.8695,0.2285,0.6049,0.3297,0.3003,0.3413,3.6910
rf,Random Forest Classifier,0.9263,0.8619,0.0725,0.7150,0.1305,0.1186,0.2098,0.6290
gbc,Gradient Boosting Classifier,0.9267,0.8578,0.2121,0.5731,0.3081,0.2781,0.3175,1.4820
et,Extra Trees Classifier,0.9242,0.8360,0.0657,0.6240,0.1175,0.1038,0.1815,0.7740
lightgbm,Light Gradient Boosting Machine,0.9264,0.8238,0.2518,0.5545,0.3452,0.3121,0.3401,1.3320
lr,Logistic Regression,0.9213,0.7949,0.0861,0.4525,0.1441,0.1226,0.1711,1.4200
ada,Ada Boost Classifier,0.9202,0.7824,0.2394,0.4683,0.3155,0.2781,0.2964,0.7090
ridge,Ridge Classifier,0.9194,0.7805,0.0397,0.2938,0.0691,0.0537,0.0840,0.4600
lda,Linear Discriminant Analysis,0.9050,0.7672,0.1204,0.2565,0.1634,0.1201,0.1300,0.5180
svm,SVM - Linear Kernel,0.9182,0.7442,0.0848,0.3698,0.1300,0.1070,0.1399,0.4960


Transformation Pipeline and Model Successfully Saved
============================================================ target_1_2
314 10000 2569 97431


,Description,Value
0,Session id,3800
1,Target,target_1_2
2,Target type,Binary
3,Original data shape,"(14061, 200)"
4,Transformed data shape,"(14061, 318)"
5,Transformed train set shape,"(9842, 318)"
6,Transformed test set shape,"(4219, 318)"
7,Numeric features,132
8,Categorical features,67
9,Rows with missing values,100.0%


...

Transformation Pipeline and Model Successfully Saved
============================================================ target_10_1


,Description,Value
0,Session id,3620
1,Target,target_10_1
2,Target type,Binary
3,Original data shape,"(100000, 201)"
4,Transformed data shape,"(100000, 321)"
5,Transformed train set shape,"(70000, 321)"
6,Transformed test set shape,"(30000, 321)"
7,Numeric features,133
8,Categorical features,67
9,Rows with missing values,100.0%


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
catboost,CatBoost Classifier,0.7253,0.7305,0.3542,0.6156,0.4496,0.2839,0.3032,13.2060
lightgbm,Light Gradient Boosting Machine,0.7233,0.7286,0.3431,0.6134,0.4400,0.2753,0.2958,7.8280
gbc,Gradient Boosting Classifier,0.7214,0.7242,0.3300,0.6120,0.4288,0.2658,0.2880,13.8550
rf,Random Forest Classifier,0.7240,0.7210,0.3020,0.6358,0.4094,0.2581,0.2885,51.2300
ada,Ada Boost Classifier,0.7159,0.7191,0.3458,0.5882,0.4354,0.2624,0.2789,7.9120
lr,Logistic Regression,0.7123,0.7061,0.2763,0.5997,0.3782,0.2230,0.2513,10.7300
ridge,Ridge Classifier,0.7152,0.7057,0.2703,0.6149,0.3755,0.2257,0.2577,9.6010
lda,Linear Discriminant Analysis,0.7142,0.7054,0.2983,0.5983,0.3980,0.2372,0.2620,7.4830
et,Extra Trees Classifier,0.7168,0.6988,0.3046,0.6055,0.4053,0.2453,0.2702,8.5120
qda,Quadratic Discriminant Analysis,0.4065,0.6850,0.9590,0.3436,0.5059,0.0739,0.1589,8.6650


Transformation Pipeline and Model Successfully Saved
CPU times: user 23min 28s, sys: 10min 10s, total: 33min 39s
Wall time: 3h 33min 38s


tcol,model,AUC,model2,AUC2,model3,AUC3
str,str,f64,str,f64,str,f64
"""target_1_1""","""CatBoost Classifier""",0.8695,"""Random Forest Classifier""",0.8619,"""Gradient Boosting Classifier""",0.8578
"""target_1_2""","""Gradient Boosting Classifier""",0.7031,"""Logistic Regression""",0.7004,"""Ridge Classifier""",0.6945
"""target_1_3""","""Logistic Regression""",0.7993,"""CatBoost Classifier""",0.7984,"""Random Forest Classifier""",0.7955
"""target_1_4""","""Logistic Regression""",0.7469,"""Random Forest Classifier""",0.7445,"""CatBoost Classifier""",0.7285
"""target_1_5""","""CatBoost Classifier""",0.7809,"""Gradient Boosting Classifier""",0.7739,"""Random Forest Classifier""",0.7635
…,…,…,…,…,…,…
"""target_9_5""","""Gradient Boosting Classifier""",0.7791,"""CatBoost Classifier""",0.7785,"""Random Forest Classifier""",0.7669
"""target_9_6""","""CatBoost Classifier""",0.6507,"""Light Gradient Boosting Machin…",0.6449,"""Random Forest Classifier""",0.6397
"""target_9_7""","""Logistic Regression""",0.6879,"""Random Forest Classifier""",0.6875,"""CatBoost Classifier""",0.6742


## Сохранение результатов

In [12]:
stats.write_csv(f"stats_caret_{limit=}_{minfrac=}.csv")

In [13]:
stats

tcol,model,AUC,model2,AUC2,model3,AUC3
str,str,f64,str,f64,str,f64
"""target_1_1""","""CatBoost Classifier""",0.8695,"""Random Forest Classifier""",0.8619,"""Gradient Boosting Classifier""",0.8578
"""target_1_2""","""Gradient Boosting Classifier""",0.7031,"""Logistic Regression""",0.7004,"""Ridge Classifier""",0.6945
"""target_1_3""","""Logistic Regression""",0.7993,"""CatBoost Classifier""",0.7984,"""Random Forest Classifier""",0.7955
"""target_1_4""","""Logistic Regression""",0.7469,"""Random Forest Classifier""",0.7445,"""CatBoost Classifier""",0.7285
"""target_1_5""","""CatBoost Classifier""",0.7809,"""Gradient Boosting Classifier""",0.7739,"""Random Forest Classifier""",0.7635
…,…,…,…,…,…,…
"""target_9_5""","""Gradient Boosting Classifier""",0.7791,"""CatBoost Classifier""",0.7785,"""Random Forest Classifier""",0.7669
"""target_9_6""","""CatBoost Classifier""",0.6507,"""Light Gradient Boosting Machin…",0.6449,"""Random Forest Classifier""",0.6397
"""target_9_7""","""Logistic Regression""",0.6879,"""Random Forest Classifier""",0.6875,"""CatBoost Classifier""",0.6742
